# Milestone 12 - Report Agent

Milestone 12 adds the standalone Report Agent. It reads existing JSON artifacts, computes KPIs, builds `final_results.json`, renders an HTML report, and prepares a compact `dashboard_payload` for a future dashboard.

## Why Standalone First

The Report Agent works alone before full workflow integration so its artifact loading, KPI calculation, HTML rendering, and masking behavior can be verified independently.

## Report Generation Pipeline

`JSON artifacts -> final_results -> HTML report -> dashboard_payload`

Mini LangGraph workflow:

`START -> report -> END`

## State Contract

The Report Agent reads `project_info`, `test_plan`, `api_results`, `bug_results`, `recommendations`, and artifact paths from State or a run directory.

It writes `final_results`, `final_results_path`, `report_result`, `report_result_path`, `report_html_path`, and `dashboard_payload`.

## Safety Note

The agent does not call `target_url`, does not start Django, and does not execute repository code. It only reads existing JSON artifacts and writes report artifacts.

Target repository: https://github.com/Vitaee/DjangoRestAPI

## Part A - Fake Report Generation

In [1]:
from IPython.display import HTML, display
from test_auto.agents.report_agent import run_report_agent_alone

fake_context = {
    "run_id": "notebook_report_demo",
    "target_url": "http://localhost:8000",
    "project_info": {
        "language": "Python",
        "framework": "Django REST Framework",
        "has_api": True,
        "has_ui": True,
        "auth_type": "JWT",
        "candidate_docs": ["README.md"],
        "candidate_api_files": ["todo/urls.py"],
        "candidate_ui_files": ["templates/login.html"],
    },
    "test_plan": {
        "scope": "JWT Todo API",
        "api_tests": [{"id": "API_001"}, {"id": "API_002"}, {"id": "API_003"}],
        "ui_tests": [],
        "performance_tests": [],
        "missing_information": ["No concrete todo id for dynamic routes."],
        "risks": ["Target app may be offline."],
    },
    "api_results": {
        "summary": {"total_tests": 3, "passed": 1, "failed": 1, "skipped": 1, "errors": 0, "pass_rate": 33.33},
        "tests": [
            {"id": "API_001", "name": "list_todos", "method": "GET", "endpoint": "/api/todos/", "status": "passed", "expected_status": 200, "actual_status": 200, "duration_ms": 11.0, "details": "ok"},
            {"id": "API_002", "name": "create_todo", "method": "POST", "endpoint": "/api/todos/", "status": "failed", "expected_status": 201, "actual_status": 400, "duration_ms": 18.0, "details": "unexpected status"},
            {"id": "API_003", "name": "get_todo_detail", "method": "GET", "endpoint": "/api/todos/<int:pk>/", "status": "skipped", "details": "dynamic parameter"},
        ],
    },
    "bug_results": {
        "summary": {"total_anomalies": 2, "high": 1, "medium": 1, "low": 0, "info": 0},
        "anomalies": [
            {"id": "BUG_001", "severity": "high", "classification": "security_risk", "title": "Authorization bypass", "source_agent": "api_testing", "recommendation": "Verify authentication and permission checks."},
            {"id": "BUG_002", "severity": "medium", "classification": "environment_error", "title": "Target unreachable", "source_agent": "api_testing", "recommendation": "Verify that the target application is running."},
        ],
        "recommendations": [
            {"priority": "high", "title": "Review authorization", "action": "Verify authentication and permission checks.", "related_anomaly_ids": ["BUG_001"]}
        ],
    },
}

result = run_report_agent_alone(context=fake_context)
result["final_results"]["kpis"], result["final_results_path"], result["report_result_path"], result["report_html_path"], result["dashboard_payload"]

({'total_api_tests': 3,
  'passed': 1,
  'failed': 1,
  'skipped': 1,
  'errors': 0,
  'pass_rate': 33.33,
  'total_ui_tests': 0,
  'ui_passed': 0,
  'ui_failed': 0,
  'ui_skipped': 0,
  'ui_errors': 0,
  'ui_pass_rate': 0.0,
  'screenshot_count': 0,
  'total_anomalies': 2,
  'high_anomalies': 1,
  'medium_anomalies': 1,
  'low_anomalies': 0,
  'info_anomalies': 0,
  'recommendation_count': 1,
  'global_score': 72.0},
 'results\\runs\\notebook_report_demo\\final_results.json',
 'results\\runs\\notebook_report_demo\\report_result.json',
 'reports\\generated\\report_notebook_report_demo.html',
 {'run_id': 'notebook_report_demo',
  'global_score': 72.0,
  'api': {'total_api_tests': 3,
   'passed': 1,
   'failed': 1,
   'skipped': 1,
   'errors': 0,
   'pass_rate': 33.33},
  'ui': {'total_ui_tests': 0,
   'passed': 0,
   'failed': 0,
   'skipped': 0,
   'errors': 0,
   'pass_rate': 0.0,
   'screenshot_count': 0},
  'bugs': {'total_anomalies': 2, 'high': 1, 'medium': 1, 'low': 0, 'info': 0}

In [2]:
display(HTML(open(result["report_html_path"], encoding="utf-8").read()))

## Part B - Run From Previous Milestone 11 Directory

This requires a previous integrated run. Replace `<run_id>` with an existing directory under `results/runs/`.

In [3]:
# from test_auto.agents.report_agent import run_report_agent_alone
# run_dir = "results/runs/<run_id>"
# previous_result = run_report_agent_alone(run_dir=run_dir)
# previous_result["final_results"]["kpis"], previous_result["report_html_path"]

## Part C - Mini LangGraph Workflow

In [4]:
from test_auto.graph.report_workflow import run_report_workflow

final_state = run_report_workflow({**fake_context, "errors": [], "agent_logs": []})
{
    "run_id": final_state["run_id"],
    "report_html_path": final_state["report_html_path"],
    "dashboard_payload": final_state["dashboard_payload"],
}

{'run_id': 'notebook_report_demo',
 'report_html_path': 'reports\\generated\\report_notebook_report_demo.html',
 'dashboard_payload': {'run_id': 'notebook_report_demo',
  'global_score': 72.0,
  'api': {'total_api_tests': 3,
   'passed': 1,
   'failed': 1,
   'skipped': 1,
   'errors': 0,
   'pass_rate': 33.33},
  'ui': {'total_ui_tests': 0,
   'passed': 0,
   'failed': 0,
   'skipped': 0,
   'errors': 0,
   'pass_rate': 0.0,
   'screenshot_count': 0},
  'bugs': {'total_anomalies': 2, 'high': 1, 'medium': 1, 'low': 0, 'info': 0},
  'recommendation_count': 1,
  'screenshot_count': 0,
  'report_html_path': 'reports\\generated\\report_notebook_report_demo.html',
  'status': 'partial'}}

## Part D - Graph Visualization

In [5]:
from test_auto.graph.report_workflow import build_report_graph

graph = build_report_graph()
try:
    display(HTML(graph.get_graph().draw_mermaid_png()))
except Exception:
    try:
        print(graph.get_graph().draw_mermaid())
    except Exception:
        print("START -> report -> END")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	report(report)
	__end__([<p>__end__</p>]):::last
	__start__ --> report;
	report --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

